# LoRA From First Principles, Measured Against Full Fine-Tuning

LLM Training Lab, module 12. Run it on a free T4: Colab (Runtime > Change runtime type > T4 GPU) or Kaggle (Settings > Accelerator > GPU T4 x2). Every cell matches a step on the lab page. Our recorded outputs are in `runs/lltLoRA/` of this repository.

Steps 1 to 4 also run on a CPU. Steps 5 to 10 need the GPU.

In [ ]:
# Runtime check. Run this first, before the install cell. It prints the menu path that fixes each problem it finds.
import importlib.metadata as md
import os, shutil, subprocess, sys

def version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return "not installed"

problems = []
if sys.version_info < (3, 11):
    problems.append(f"Python {sys.version.split()[0]} is too old. transformers 5.22.1 needs 3.11 or later; 3.12 recommended.")
try:
    gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:
    gpus = []
print("GPUs:", gpus or "none")
if not gpus:
    problems.append("No GPU. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2. Steps 1 to 4 still run on a CPU.")
elif any("P100" in g for g in gpus):
    problems.append("P100 found. Current PyTorch wheels dropped it. Kaggle: Settings > Accelerator > GPU T4 x2.")
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
disk = shutil.disk_usage(".").free / 1e9
print(f"python {sys.version.split()[0]} | torch {version('torch')} | transformers {version('transformers')} | "
      f"peft {version('peft')} | RAM {ram:.0f} GB | free disk {disk:.0f} GB")
if ram < 12:
    problems.append("Under 12 GB of RAM. Step 7 keeps two copies of the weights on the CPU.")
for p in problems or ["Runtime OK"]:
    print(p)


Install the pinned Hugging Face libraries. Keep the PyTorch and NumPy your runtime already has. Kaggle's image ships torchao 0.10, which PEFT 0.21.2 refuses, so the cell removes it.

In [ ]:
%pip uninstall -y -q torchao
%pip install -q transformers==5.22.1 tokenizers==0.23.3 huggingface_hub==2.4.0 peft==0.21.2 accelerate==1.15.0 bitsandbytes==0.50.2 datasets==5.1.0


In [ ]:
%%writefile gpt_model.py
# gpt_model.py - the GPT from the lab "Assemble a Modern GPT" (steps 2 to 4), definitions only.
import math
from dataclasses import dataclass, replace

import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Config:
    vocab: int = 50257
    ctx: int = 1024
    d: int = 768
    layers: int = 12
    heads: int = 12
    norm: str = "rmsnorm"   # "rmsnorm" or "layernorm"
    mlp: str = "swiglu"     # "swiglu" or "gelu"
    pos: str = "rope"       # "rope" or "learned"
    prenorm: bool = True
    bias: bool = False
    eps: float = 1e-5

GPT2 = Config(norm="layernorm", mlp="gelu", pos="learned", bias=True)
MODERN = Config()

class RMSNorm(nn.Module):
    def __init__(self, d, eps):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

def make_norm(cfg):
    if cfg.norm == "rmsnorm":
        return RMSNorm(cfg.d, cfg.eps)
    return nn.LayerNorm(cfg.d, eps=cfg.eps, bias=cfg.bias)


def rope_tables(T, hd, offset=0, base=10000.0, device=None):
    inv = base ** (-torch.arange(0, hd, 2, device=device).float() / hd)
    ang = torch.outer(torch.arange(offset, offset + T, device=device).float(), inv)
    return ang.cos(), ang.sin()

def apply_rope(x, cos, sin):
    # x is (batch, heads, T, head_dim). Rotate the pairs (x[2i], x[2i+1]) as in step 1.
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.heads, self.rope = cfg.heads, cfg.pos == "rope"
        self.qkv = nn.Linear(cfg.d, 3 * cfg.d, bias=cfg.bias)
        self.proj = nn.Linear(cfg.d, cfg.d, bias=cfg.bias)

    def forward(self, x, cos=None, sin=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.heads, C // self.heads).transpose(1, 2) for t in (q, k, v))
        if self.rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)   # queries and keys, never values
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

def swiglu_hidden(d):
    # 8d/3 keeps the parameter count of a 4d GELU MLP; round up to a multiple of 64
    return 64 * math.ceil(8 * d / 3 / 64)

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.swiglu = cfg.mlp == "swiglu"
        hidden = swiglu_hidden(cfg.d) if self.swiglu else 4 * cfg.d
        self.gate = nn.Linear(cfg.d, hidden, bias=cfg.bias) if self.swiglu else None
        self.up = nn.Linear(cfg.d, hidden, bias=cfg.bias)
        self.down = nn.Linear(hidden, cfg.d, bias=cfg.bias)

    def forward(self, x):
        if self.swiglu:
            return self.down(F.silu(self.gate(x)) * self.up(x))
        return self.down(F.gelu(self.up(x), approximate="tanh"))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.prenorm = cfg.prenorm
        self.n1, self.n2 = make_norm(cfg), make_norm(cfg)
        self.attn, self.mlp = Attention(cfg), MLP(cfg)

    def forward(self, x, cos=None, sin=None):
        if self.prenorm:
            x = x + self.attn(self.n1(x), cos, sin)
            return x + self.mlp(self.n2(x))
        x = self.n1(x + self.attn(x, cos, sin))
        return self.n2(x + self.mlp(x))


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.vocab, cfg.d)
        self.pos = nn.Embedding(cfg.ctx, cfg.d) if cfg.pos == "learned" else None
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.layers)])
        self.norm = make_norm(cfg)
        self.head = nn.Linear(cfg.d, cfg.vocab, bias=False)
        self.head.weight = self.tok.weight   # tie first, so the init below sets the shared tensor once
        for name, p in self.named_parameters():
            if name.endswith(("proj.weight", "down.weight")):
                # the two layers per block that write into the residual stream
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.layers))
            elif p.dim() == 2:
                nn.init.normal_(p, 0.0, 0.02)
            elif name.endswith("bias"):
                nn.init.zeros_(p)

    def forward(self, idx, offset=0):
        B, T = idx.shape
        x = self.tok(idx)
        cos = sin = None
        if self.pos is None:
            cos, sin = rope_tables(T, self.cfg.d // self.cfg.heads, offset, device=idx.device)
        else:
            x = x + self.pos(torch.arange(offset, offset + T, device=idx.device))
        for block in self.blocks:
            x = block(x, cos, sin)
        return self.head(self.norm(x))

# The model our TinyStories pretraining run uses (lltTinyStories)
TINY = replace(MODERN, vocab=4096, ctx=512, d=320, layers=8, heads=8)


In [ ]:
%%writefile lora_lab.py
# lora_lab.py - data, training loop and evaluation for the lltLoRA lab.
# The notebook writes this file; our recorded runs (kaggle/lora_worker.py) import the same file.
import math
import re
import time

import torch
from datasets import load_dataset

MODEL = "Qwen/Qwen3.5-0.8B-Base"
PROMPT = """Question: {q}
Answer:"""


def load_gsm8k(split, n=None):
    rows = load_dataset("openai/gsm8k", "main", split=split)
    if n is not None:
        rows = rows.select(range(n))
    # Drop the calculator notes such as <<48/2=24>> from each worked answer.
    return [(PROMPT.format(q=r["question"]), " " + re.sub("<<[^>]*>>", "", r["answer"])) for r in rows]


def encode(tok, pairs):
    rows = []
    for prompt, answer in pairs:
        p = tok(prompt)["input_ids"]
        a = tok(answer)["input_ids"] + [tok.eos_token_id]
        rows.append((p + a, [-100] * len(p) + a))      # loss only on the answer and its end token
    return rows


def text_rows(tok, n_rows=32, T=512):
    # General text to measure forgetting: Wikipedia articles the task never shows.
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    ids = tok("".join(ds["text"]))["input_ids"][: n_rows * T]
    return [(ids[i:i + T], ids[i:i + T]) for i in range(0, n_rows * T, T)]


def micro_batches(rows, budget):
    # Group rows so that rows x longest row stays within budget tokens. Memory follows that product,
    # because the logits tensor is rows x length x 248,320 vocabulary entries.
    rows = sorted(rows, key=lambda r: len(r[0]))
    groups, cur = [], []
    for r in rows:
        if cur and (len(cur) + 1) * len(r[0]) > budget:
            groups.append(cur)
            cur = []
        cur.append(r)
    return groups + [cur]


def collate(rows, pad_id, device):
    T = max(len(ids) for ids, _ in rows)
    ids = torch.full((len(rows), T), pad_id, dtype=torch.long)
    labels = torch.full((len(rows), T), -100, dtype=torch.long)
    mask = torch.zeros((len(rows), T), dtype=torch.long)
    for i, (x, y) in enumerate(rows):
        ids[i, :len(x)] = torch.tensor(x)
        labels[i, :len(y)] = torch.tensor(y)
        mask[i, :len(x)] = 1
    return ids.to(device), mask.to(device), labels.to(device)


def amp(device):
    return torch.autocast(device, dtype=torch.float16, enabled=device == "cuda")


@torch.no_grad()
def mean_loss(model, rows, pad_id, device, budget=2048):
    # Mean next-token loss over every label that is not -100, in nats per token.
    was_training = model.training
    model.eval()
    total, n = 0.0, 0
    for group in micro_batches(rows, budget):
        ids, mask, labels = collate(group, pad_id, device)
        k = int((labels[:, 1:] != -100).sum())
        with amp(device):
            loss = model(input_ids=ids, attention_mask=mask, labels=labels, num_items_in_batch=k).loss
        total += loss.item() * k
        n += k
    model.train(was_training)
    return total / n


def train(model, rows, steps, opt, pad_id, device="cuda", batch=16, budget=1024, warmup=10, seed=0,
          on_step=None):
    params = [p for p in model.parameters() if p.requires_grad]
    # Linear warmup over the first warmup steps, then linear decay to 0 at the last step.
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warmup) * (1 - s / steps))
    # Qwen3.5's fp16 gradients overflow at the default scale of 65,536, so start lower.
    scaler = torch.amp.GradScaler(device, init_scale=2048, enabled=device == "cuda")
    g = torch.Generator().manual_seed(seed)
    order, history = [], []
    model.train()
    for step in range(steps):
        t0 = time.time()
        if len(order) < batch:
            order += torch.randperm(len(rows), generator=g).tolist()
        picked = [rows[i] for i in order[:batch]]
        order = order[batch:]
        n_tok = sum(sum(1 for t in y if t != -100) for _, y in picked)
        loss_sum = 0.0
        for group in micro_batches(picked, budget):
            ids, mask, labels = collate(group, pad_id, device)
            with amp(device):
                # num_items_in_batch makes the loss a sum over this micro-batch divided by the
                # answer tokens of the whole batch, so every token weighs the same.
                loss = model(input_ids=ids, attention_mask=mask, labels=labels, num_items_in_batch=n_tok).loss
            scaler.scale(loss).backward()
            loss_sum += loss.item()
        scale = scaler.get_scale()
        scaler.unscale_(opt)
        gnorm = float(torch.nn.utils.clip_grad_norm_(params, 1.0))
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        sched.step()
        if device == "cuda":
            torch.cuda.synchronize()
        rec = {"step": step + 1, "trainLoss": round(loss_sum, 5), "gradNorm": round(gnorm, 4),
               "lr": sched.get_last_lr()[0], "skipped": scaler.get_scale() < scale,
               "sec": round(time.time() - t0, 3), "tokens": n_tok}
        history.append(rec)
        if on_step:
            on_step(rec)
    return history


NUMBER = "-?[0-9][0-9,]*(?:[.][0-9]+)?"


def final_number(text, strict=True):
    # strict: the number after ####, the answer format of the training data.
    # lenient: that number, or else the last number before the model starts a new question.
    text = text.split("Question:")[0]
    m = re.search("####[ ]*(" + NUMBER + ")", text)
    if m:
        return m.group(1).replace(",", "").rstrip(".")
    if strict:
        return None
    found = re.findall(NUMBER, text)
    return found[-1].replace(",", "").rstrip(".") if found else None


@torch.no_grad()
def accuracy(model, tok, pairs, device="cuda", batch=50, max_new=256):
    # Greedy answers; a problem counts as solved when the number after #### matches the reference.
    was_training = model.training
    model.eval()
    tok.padding_side = "left"
    strict, lenient, outputs = 0, 0, []
    for i in range(0, len(pairs), batch):
        chunk = pairs[i:i + batch]
        enc = tok([p for p, _ in chunk], return_tensors="pt", padding=True).to(device)
        with amp(device):
            out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True,
                                 pad_token_id=tok.pad_token_id)
        for (_, ref), seq in zip(chunk, out[:, enc["input_ids"].shape[1]:]):
            text = tok.decode(seq, skip_special_tokens=True)
            gold = final_number(ref)
            strict += final_number(text) == gold
            lenient += final_number(text, strict=False) == gold
            outputs.append(text)
    tok.padding_side = "right"
    model.train(was_training)
    return strict / len(pairs), lenient / len(pairs), outputs


## Step 1. Write a LoRA linear layer

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # Kaggle has 2 GPUs; this lab uses one
import math
import torch
import torch.nn as nn

torch.set_num_threads(4)

class LoRALinear(nn.Module):
    # y = W x + scale * B A x, with W frozen and only A and B trained.
    def __init__(self, base, r=16, alpha=32, rslora=False, dropout=0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)                      # freeze W (and its bias, if any)
        self.r = r
        self.scale = alpha / math.sqrt(r) if rslora else alpha / r
        w = base.weight
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=w.device, dtype=w.dtype))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=w.device, dtype=w.dtype))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))  # PEFT's default init for A; B stays 0
        self.drop = nn.Dropout(dropout)
        self.merged = False

    def forward(self, x):
        y = self.base(x)
        if self.merged:
            return y                                      # W already holds W + scale * B A
        return y + (self.drop(x) @ self.A.T @ self.B.T) * self.scale

    @torch.no_grad()
    def merge(self):
        if not self.merged:
            self.base.weight += (self.B @ self.A) * self.scale
            self.merged = True

    @torch.no_grad()
    def unmerge(self):
        if self.merged:
            self.base.weight -= (self.B @ self.A) * self.scale
            self.merged = False

torch.manual_seed(0)
base = nn.Linear(1024, 3584, bias=False)                 # the shape of one MLP up_proj in Qwen3.5-0.8B
x = torch.randn(4, 10, 1024)
y0 = base(x).detach()
layer = LoRALinear(base, r=16, alpha=32)
print("output unchanged at init:", torch.equal(layer(x), y0))
n_train = sum(p.numel() for p in layer.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in layer.parameters() if not p.requires_grad)
print(f"trainable {n_train:,} | frozen {n_frozen:,} | {100 * n_train / n_frozen:.2f}% of the layer")

layer(x).pow(2).mean().backward()
print("grad on W:", base.weight.grad, "| grad on A is all zero:", bool(layer.A.grad.abs().max() == 0),
      "| grad on B is all zero:", bool(layer.B.grad.abs().max() == 0))
for r in [1, 4, 16, 64, 256]:
    print(f"r = {r:>3}  alpha/r = {32 / r:7.4f}  rsLoRA alpha/sqrt(r) = {32 / math.sqrt(r):7.4f}")


## Step 2. Merge and unmerge

In [ ]:
torch.manual_seed(1)
with torch.no_grad():
    layer.B.normal_(0, 0.02)                             # stand-in for a trained B, so the update is not zero
w_before = base.weight.detach().clone()
y_adapter = layer(x).detach()

layer.merge()
y_merged = layer(x).detach()
print("merged output matches the adapter output:", torch.allclose(y_merged, y_adapter, atol=1e-5),
      f"| max abs diff {(y_merged - y_adapter).abs().max():.2e}")
delta = base.weight.detach() - w_before
print("rank of the merged update:", int(torch.linalg.matrix_rank(delta)), "of a possible", min(delta.shape))

layer.unmerge()
print("unmerge restores W:", torch.allclose(base.weight, w_before, atol=1e-6),
      f"| max abs diff {(base.weight - w_before).abs().max():.2e}")


## Step 3. Add LoRA to your GPT

In [ ]:
from gpt_model import GPT, TINY    # your GPT from the lab Assemble a Modern GPT

def add_lora(model, targets, r=8, alpha=16, rslora=False):
    # Freeze every weight, then wrap each nn.Linear whose name is in targets.
    for p in model.parameters():
        p.requires_grad_(False)
    for module in list(model.modules()):
        for name, child in list(module.named_children()):
            if isinstance(child, nn.Linear) and name in targets:
                setattr(module, name, LoRALinear(child, r, alpha, rslora))
    return model

def count(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, sum(p.numel() for p in model.parameters())

idx = torch.randint(0, TINY.vocab, (2, 64), generator=torch.Generator().manual_seed(0))
for label, targets in [("attention only", {"qkv", "proj"}),
                       ("all linear layers", {"qkv", "proj", "gate", "up", "down"})]:
    torch.manual_seed(0)
    model = GPT(TINY).eval()
    with torch.no_grad():
        before = model(idx)
    add_lora(model, targets, r=8, alpha=16)
    with torch.no_grad():
        after = model(idx)
    trainable, total = count(model)
    print(f"{label:17}  trainable {trainable:>7,} of {total:,} ({100 * trainable / total:.2f}%)  "
          f"logits unchanged: {torch.equal(before, after)}")
print("head stays tied and frozen:", model.head.weight is model.tok.weight, model.head.weight.requires_grad)


## Step 4. Check your layer against PEFT on Qwen3.5-0.8B-Base

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from lora_lab import MODEL, load_gsm8k, encode, collate

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).eval()
ids, mask, labels = collate(encode(tok, load_gsm8k("train", 2)), tok.pad_token_id, "cpu")
batch = dict(input_ids=ids, attention_mask=mask, labels=labels)

cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", task_type="CAUSAL_LM")
peft_model = get_peft_model(model, cfg)
torch.manual_seed(1)
with torch.no_grad():
    for name, p in peft_model.named_parameters():
        if "lora_B" in name:
            p.normal_(0, 0.02)                           # non-zero B, so the check exercises the adapters
    ref = peft_model(**batch)
n_peft = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
adapters = {name: (m.lora_A["default"].weight.clone(), m.lora_B["default"].weight.clone())
            for name, m in peft_model.base_model.model.named_modules() if hasattr(m, "lora_A")}
print(len(adapters), "layers wrapped by PEFT:", sorted({n.split(".")[-1] for n in adapters}))

model = peft_model.unload()                              # the base model again, weights untouched
add_lora(model, {n.split(".")[-1] for n in adapters}, r=16, alpha=32)
with torch.no_grad():
    for name, m in model.named_modules():
        if isinstance(m, LoRALinear):
            m.A.copy_(adapters[name][0])
            m.B.copy_(adapters[name][1])
    ours = model(**batch)
n_ours = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable  ours {n_ours:,}  PEFT {n_peft:,}")
print(f"loss       ours {ours.loss.item():.6f}  PEFT {ref.loss.item():.6f}")
print(f"max abs logit difference {(ours.logits - ref.logits).abs().max():.2e}")

for m in model.modules():
    if isinstance(m, LoRALinear):
        m.merge()
with torch.no_grad():
    merged = model(**batch)
print(f"after merge, loss {merged.loss.item():.6f}, max abs logit difference {(merged.logits - ref.logits).abs().max():.2e}")
del model, peft_model, ref, ours, merged                # frees about 3 GB of RAM before the GPU steps


## Step 5. Full fine-tuning baseline

In [ ]:
FULL_LR = 1e-5        # best held-out loss of 3e-6, 1e-5, 3e-5 and 1e-4 in our sweep
LORA_LR = 3e-4        # best held-out loss of 3e-5, 1e-4, 3e-4, 1e-3 and 3e-3, r16 on all linear layers
FULL_BUDGET = 1024    # tokens per micro-batch (rows x longest row) for full fine-tuning
LORA_BUDGET = 2048    # LoRA keeps no gradients or optimizer state for the base, so it fits twice the tokens


In [ ]:
import random
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer
from lora_lab import MODEL, load_gsm8k, encode, text_rows, mean_loss, train, accuracy

DEV = "cuda"
tok = AutoTokenizer.from_pretrained(MODEL)
pairs = load_gsm8k("train")
random.Random(1234).shuffle(pairs)
train_rows = encode(tok, pairs[:2000])                   # 2,000 worked answers, 125 steps of 16
test = load_gsm8k("test")
held, wiki, problems = encode(tok, test[:300]), text_rows(tok), test[:200]

for name in ["model", "opt"]:                          # running this cell again for another FULL_LR:
    globals().pop(name, None)                            # free the last run's model and optimizer first
torch.cuda.empty_cache()

def load():
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32, device_map=DEV)
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    return model

def report(name, model, hist=None):
    acc, _, _ = accuracy(model, tok, problems, DEV)
    line = (f"{name:22} held-out loss {mean_loss(model, held, tok.pad_token_id, DEV):.4f} | "
            f"wiki loss {mean_loss(model, wiki, tok.pad_token_id, DEV):.4f} | GSM8K {acc:.1%}")
    if hist:
        line += (f" | peak {torch.cuda.max_memory_allocated() / 1e9:.2f} GB | "
                 f"{sum(h['sec'] for h in hist) / 60:.1f} min | skipped steps {sum(h['skipped'] for h in hist)}")
    print(line)

model = load()
N = sum(p.numel() for p in model.parameters())
print(f"{N:,} parameters | predicted training state: 8-bit AdamW {10 * N / 1e9:.2f} GB, 32-bit AdamW {16 * N / 1e9:.2f} GB")
report("base model", model)

W0 = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters()}   # for the SVD in step 7
opt = bnb.optim.AdamW8bit(model.parameters(), lr=FULL_LR, weight_decay=0.0)
torch.cuda.reset_peak_memory_stats()
hist = train(model, train_rows, 125, opt, tok.pad_token_id, DEV, budget=FULL_BUDGET)
report("full fine-tune", model, hist)


## Step 6. Sweep the full fine-tune's learning rate

Change `FULL_LR` in the settings cell and run step 5 again, once per rate. Our runs used 3e-6, 1e-5, 3e-5 and 1e-4.

## Step 7. The SVD of the full fine-tune's update

In [ ]:
del opt
torch.cuda.empty_cache()
params = dict(model.named_parameters())
linear = [n for n, m in model.named_modules() if isinstance(m, torch.nn.Linear) and not n.endswith("lm_head")]
energy, tops = {}, {}
with torch.no_grad():
    for n in linear:
        delta = params[n + ".weight"] - W0[n + ".weight"].to(DEV)          # what full fine-tuning changed
        U, S, Vh = torch.linalg.svd(delta, full_matrices=False)
        energy[n] = ((S ** 2).cumsum(0) / (S ** 2).sum()).cpu()            # energy in the top 1, 2, 3, ... values
        tops[n] = (U[:, :256].cpu(), S[:256].cpu(), Vh[:256].cpu())

print(len(linear), "linear layers. Median share of each update's energy in its top r singular values:")
for r in [1, 4, 16, 64, 256]:
    print(f"  r = {r:>3}  {torch.tensor([float(e[min(r, len(e)) - 1]) for e in energy.values()]).median():.3f}")
need = sorted(int((e < 0.9).sum()) + 1 for e in energy.values())
print("singular values needed for 90% of the energy, median", need[len(need) // 2], "| max", need[-1])

# Rebuild the model from the base plus a rank-r piece of every linear update. Embeddings and norms go back
# to the base, because LoRA on all linear layers cannot change them.
finetuned = {n: p.detach().to("cpu", copy=True) for n, p in params.items()}
for r in [1, 4, 16, 64, 256, "all"]:
    with torch.no_grad():
        for n, p in params.items():
            if n.endswith(".weight") and n[:-7] in tops:
                U, S, Vh = tops[n[:-7]]
                if r == "all":
                    p.copy_(finetuned[n])
                else:
                    p.copy_(W0[n] + (U[:, :r] * S[:r]) @ Vh[:r])
            else:
                p.copy_(W0[n])
    print(f"rank {r!s:>3} slice of every update  held-out loss {mean_loss(model, held, tok.pad_token_id, DEV):.4f}"
          f" | wiki loss {mean_loss(model, wiki, tok.pad_token_id, DEV):.4f}")


## Step 8. LoRA on all linear layers

In [ ]:
import os
from peft import LoraConfig, get_peft_model

del model, params, finetuned, W0, tops
torch.cuda.empty_cache()

def lora_run(r=16, targets="all-linear", lr=LORA_LR, seed=0, save=None):
    model = load()
    model.enable_input_require_grads()                   # gradient checkpointing needs a grad on the inputs
    torch.manual_seed(seed)                              # sets the random init of every A
    cfg = LoraConfig(r=r, lora_alpha=32, lora_dropout=0.0, target_modules=targets, task_type="CAUSAL_LM")
    model = get_peft_model(model, cfg)
    trainable = [p for p in model.parameters() if p.requires_grad]
    print(f"r = {r}, {sum(p.numel() for p in trainable):,} trainable parameters")
    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=0.0)
    torch.cuda.reset_peak_memory_stats()
    hist = train(model, train_rows, 125, opt, tok.pad_token_id, DEV, budget=LORA_BUDGET, seed=seed)
    report(f"LoRA r{r} " + (targets if isinstance(targets, str) else "attention"), model, hist)
    if save:
        model.save_pretrained(save)                      # writes the adapter only
        print(save, sum(os.path.getsize(os.path.join(save, f)) for f in os.listdir(save)) / 1e6, "MB")
    return model

model = lora_run(save="lora-r16")


## Step 9. Sweep the LoRA learning rate

Call `lora_run(lr=...)` once per rate, for example `m = lora_run(lr=1e-4)`, then `del m` and `torch.cuda.empty_cache()`. Our runs used 3e-5, 1e-4, 3e-4, 1e-3 and 3e-3.

## Step 10. Rank and target layers

In [ ]:
del model
torch.cuda.empty_cache()
ATTENTION = ["q_proj", "k_proj", "v_proj", "o_proj", "in_proj_qkv", "in_proj_z", "in_proj_a", "in_proj_b", "out_proj"]

# Each run takes about as long as the one in step 8. Add the settings you want to test.
for r, targets in [(1, "all-linear"), (16, ATTENTION)]:
    m = lora_run(r=r, targets=targets)
    del m
    torch.cuda.empty_cache()
